# Exploratory Data Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import linalg
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import normalize

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

import glob

In [ ]:
files = glob.glob('../data/eda/*.csv')
df = pd.concat([pd.read_csv(f, dtype={
    'start_station_id': 'str',
    'end_station_id': 'str',
    'member_casual': 'category',
    'rideable_type': 'category'
}) for f in files], ignore_index=True)

print(df.shape)
df.head()

In [ ]:
# drop missing station IDs
df = df.dropna(subset=['start_station_id', 'end_station_id'])

# parse timestamps
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])

# extract hour and day of week for later
df['hour'] = df['started_at'].dt.hour
df['day_of_week'] = df['started_at'].dt.dayofweek  # 0=Monday, 6=Sunday

print(df.shape)
print(df.dtypes)

In [ ]:
# count total departures per station
departure_counts = df['start_station_id'].value_counts()

# look at the distribution first
print(departure_counts.describe())
print(f"\nStations with < 100 trips: {(departure_counts < 100).sum()}")
print(f"Stations with < 500 trips: {(departure_counts < 500).sum()}")
print(f"Total stations: {len(departure_counts)}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.hist(departure_counts, bins=50, log=True)
plt.xlabel("Total departures")
plt.ylabel("Number of stations (log scale)")
plt.title("Distribution of station activity")
plt.axvline(departure_counts.quantile(0.25), color='red', linestyle='--', label='25th percentile')
plt.legend()
plt.show()

In [ ]:
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix

# 1. build count matrix on unfiltered data
all_stations = pd.concat([df['start_station_id'], df['end_station_id']]).unique()
n = len(all_stations)
station_to_idx = {s: i for i, s in enumerate(all_stations)}

C = np.zeros((n, n))
np.add.at(C, (
    df['start_station_id'].map(station_to_idx).values,
    df['end_station_id'].map(station_to_idx).values
), 1)

# 2. find strongly connected components
n_components, labels = connected_components(csr_matrix(C), directed=True, connection='strong')
print(f"Number of strongly connected components: {n_components}")

# 3. keep only stations in the largest component
component_sizes = pd.Series(labels).value_counts()
print(component_sizes.head(10))
largest_component = component_sizes.idxmax()

stations_to_keep = [
    all_stations[i] 
    for i, label in enumerate(labels) 
    if label == largest_component
]
print(f"Stations in main component: {len(stations_to_keep)}")
print(f"Stations dropped: {n - len(stations_to_keep)}")

# 4. filter dataframe
df = df[
    df['start_station_id'].isin(stations_to_keep) & 
    df['end_station_id'].isin(stations_to_keep)
]
print(f"Trips remaining: {len(df):,}")

In [ ]:
# re-encode with filtered stations
all_stations = pd.concat([df['start_station_id'], df['end_station_id']]).unique()
n = len(all_stations)

station_to_idx = {s: i for i, s in enumerate(all_stations)}
idx_to_station = {i: s for s, i in station_to_idx.items()}

df['start_idx'] = df['start_station_id'].map(station_to_idx)
df['end_idx'] = df['end_station_id'].map(station_to_idx)

print(f"Final network: {n} stations, {len(df):,} trips")

In [ ]:
C = np.zeros((n, n))
np.add.at(C, (df['start_idx'].values, df['end_idx'].values), 1)

# normalize rows
row_sums = C.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
P = C / row_sums

print(f"Transition matrix shape: {P.shape}")
print(f"Row sums: {P.sum(axis=1).min():.4f} to {P.sum(axis=1).max():.4f}")

In [ ]:
from scipy import linalg

eigenvalues, eigenvectors = linalg.eig(P.T)

# find eigenvector with eigenvalue closest to 1
idx = np.argmin(np.abs(eigenvalues - 1))
pi = eigenvectors[:, idx].real
pi = pi / pi.sum()

print(f"Sums to: {pi.sum():.6f}")
print(f"Min value: {pi.min():.6f}")
print(f"Max value: {pi.max():.6f}")

In [ ]:
# map back to station names
station_pi = pd.Series(pi, index=[idx_to_station[i] for i in range(n)])
station_pi = station_pi.sort_values(ascending=False)

print("Top 10 stations:")
print(station_pi.head(10))
print("\nBottom 10 stations:")
print(station_pi.tail(10))

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(sorted(pi, reverse=True))
plt.xlabel("Station rank")
plt.ylabel("π_i")
plt.title("Stationary distribution across stations")
plt.show()

In [ ]:
# build global count matrix once outside the loop
C_global = np.zeros((n, n))
np.add.at(C_global, (df['start_idx'].values, df['end_idx'].values), 1)

alpha = 0.01
pi_by_hour = {}

for hour in range(24):
    df_hour = df[df['hour'] == hour]
    
    C_h = np.zeros((n, n))
    np.add.at(C_h, (df_hour['start_idx'].values, df_hour['end_idx'].values), 1)
    
    # blend hourly counts with global prior
    C_smoothed = C_h + alpha * C_global
    
    row_sums = C_smoothed.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    P_h = C_smoothed / row_sums
    
    eigenvalues, eigenvectors = linalg.eig(P_h.T)
    idx = np.argmin(np.abs(eigenvalues - 1))
    pi_h = eigenvectors[:, idx].real
    pi_h = pi_h / pi_h.sum()
    
    pi_by_hour[hour] = pi_h

print("Done")

In [ ]:
# pick your top station from earlier
station = station_pi.index[0]
station_idx = station_to_idx[station]

hourly_pi = [pi_by_hour[h][station_idx] for h in range(24)]

plt.figure(figsize=(10, 4))
plt.plot(range(24), hourly_pi, marker='o')
plt.xlabel("Hour of day")
plt.ylabel("π_i")
plt.title(f"Station {station} — stationary distribution by hour")
plt.xticks(range(24))
plt.show()

In [ ]:
# find the station name
station_name = df[df['start_station_id'] == '6140.05']['start_station_name'].iloc[0]
print(station_name)

In [ ]:
# build station id to name mapping
station_id_to_name = dict(zip(df['start_station_id'], df['start_station_name']))

# also grab lat/lng while you're at it — you'll need these for the map
station_id_to_lat = dict(zip(df['start_station_id'], df['start_lat']))
station_id_to_lng = dict(zip(df['start_station_id'], df['start_lng']))

# verify
print(station_id_to_name['6140.05'])

idx_to_info = {
    idx: {
        'name': station_id_to_name.get(station_id),
        'lat': station_id_to_lat.get(station_id),
        'lng': station_id_to_lng.get(station_id)
    }
    for idx, station_id in idx_to_station.items()
}

# verify
print(idx_to_info[station_to_idx['6140.05']])
# {'name': 'W 21 St & 6 Ave', 'lat': 40.742..., 'lng': -73.994...}

In [ ]:
import json

# save pi matrix as numpy
pi_matrix = np.array([pi_by_hour[h] for h in range(24)])
np.save('../outputs/pi_by_hour.npy', pi_matrix)

# save station info as json
# need to convert to serializable format first
station_info = {
    str(idx): {
        'name': info['name'],
        'lat': float(info['lat']) if info['lat'] is not None else None,
        'lng': float(info['lng']) if info['lng'] is not None else None
    }
    for idx, info in idx_to_info.items()
}

with open('../outputs/stations.json', 'w') as f:
    json.dump(station_info, f)

print("Saved!")

In [ ]:
# in notebook — compute and save flow ratio
arrivals   = C_global.sum(axis=0)   # column sums
departures = C_global.sum(axis=1)   # row sums
flow_ratio = arrivals / (departures + 1e-10)

np.save('../outputs/flow_ratio.npy', flow_ratio)
print("Saved flow_ratio.npy")

In [ ]:
# compute flow ratio from global count matrix
arrivals   = C_global.sum(axis=0)   # column sums — bikes coming in
departures = C_global.sum(axis=1)   # row sums — bikes going out
flow_ratio = arrivals / (departures + 1e-10)
flow_ratio_centered = flow_ratio / np.median(flow_ratio)
np.save('../outputs/flow_ratio.npy', flow_ratio_centered)

print(f"Stations > 1: {(flow_ratio_centered > 1).sum()}")
print(f"Stations < 1: {(flow_ratio_centered < 1).sum()}")

In [ ]:
# in notebook — compute hourly flow ratio
arrivals_by_hour   = np.zeros((24, n))   # arrivals per hour per station
departures_by_hour = np.zeros((24, n))

for h in range(24):
    mask = df['hour'] == h
    np.add.at(arrivals_by_hour[h],   df.loc[mask, 'end_idx'].values,   1)
    np.add.at(departures_by_hour[h], df.loc[mask, 'start_idx'].values, 1)

flow_ratio_by_hour = arrivals_by_hour / (departures_by_hour + 1e-10)
np.save('../outputs/flow_ratio_by_hour.npy', flow_ratio_by_hour)